In [1]:
import os
from pathlib import Path
# Change cwd to the project root (parent of 'notebooks/')
os.chdir(Path.cwd().parent)
Path.cwd()

PosixPath('/Users/jbrandt/code/birddog')

In [2]:
from copy import copy, deepcopy
from urllib.parse import quote, unquote, urlparse
import mwparserfromhell

import json
from birddog.core import (
    Archive, 
    )
from birddog.wiki import (
    ARCHIVE_BASE,
    WIKI_NAMESPACE,
    mw_read_page,
    mw_page_doc_url,
    lookup_namespace_id,
    get_title,
    check_page_changes,
    expand_link_target,
    _read_wiki_text,
    _check_page_existence_chunked,
    _parse_wikitext_table,
    _is_table,
    batch_fetch_document_links,
    API_URL,
    canonicalize_title,
    _is_category_link,
    page_label,
    parent_title,
    )


2026-06-04 07:33:18,022 [INFO] Using local folder /Users/jbrandt/code/birddog/.cache for storage.
2026-06-04 07:33:18,174 [INFO] Translation is enabled. Using GCP translator
2026-06-04 07:33:18,174 [INFO] Using Google Cloud translation API
2026-06-04 07:33:18,175 [INFO] GoogleCloudTranslator using REST API


In [3]:
page_label("ДІСЗМО")

'DISZMO'

In [4]:
#parent_title("ДІСЗМО")

In [ ]:
"Архів:ЦДАЗУ"

In [5]:
pg = mw_read_page("Архів:ЦДАГОУ")  "

2026-06-04 07:33:21,237 [INFO] 
service throttle report:
  host_key                            cfg_rps  act_rps   tokens  blocked_s max_in_flight
  ----------------------------------------------------------------------------------------
  uk.wikisource.org:api                  4.00     3.35     4.00       0.00            4


In [6]:
pg["tables"]

[{'header': [{'uk': 'Номер фонду'},
   {'uk': 'Назва'},
   {'uk': 'Крайні дати'},
   {'uk': 'Справ'}],
  'children': [[{'text': {'uk': '1', 'en': '1'},
     'link': '/wiki/Архів:ЦДАГОУ/1',
     'exists': True},
    {'text': {'uk': 'Центральний комітет Комуністичної партії України (ЦК КПУ), м. Київ (1918–1991)'},
     'link': None},
    {'text': {'uk': '1917–1991', 'en': '1917–1991'}, 'link': None},
    {'text': {'uk': '166299', 'en': '166299'}, 'link': None}],
   [{'text': {'uk': '5', 'en': '5'},
     'link': '/wiki/Архів:ЦДАГОУ/5',
     'exists': False},
    {'text': {'uk': 'Комісія з історії громадянської війни при ЦК КП(б)У, м. Київ (1931–1938)'},
     'link': None},
    {'text': {'uk': '1917–1938', 'en': '1917–1938'}, 'link': None}],
   [{'text': {'uk': '6', 'en': '6'},
     'link': '/wiki/Архів:ЦДАГОУ/6',
     'exists': False},
    {'text': {'uk': 'Центральний комітет Комуністичної партії Західної України (ЦК КПЗУ) (1923–1938)'},
     'link': None},
    {'text': {'uk': '1917–1936'

In [ ]:
mw_page_doc_url(pg)

In [ ]:
txt = _read_wiki_text("Архів:ДАПО/222/1/682")
txt

In [ ]:
def get_links(title):
    params = {
        "action": "parse",
        "prop": "links|iwlinks|externallinks",
        "format": "json",
        "page": canonicalize_title(title),
    }
    return fetch_url(API_URL, params=params, json=True)

In [ ]:
#get_links("Архів:ДАДнО/К/Ф")

In [ ]:
titles = [ 
    "Архів:ДАПО/978/1",
    "Архів:ДАЖО/1/74",
    "Архів:ЦДІАК/28/1",
    "Архів:ДАЖО/1",
    "Архів:ДАК/Р-352",
    "Архів:Архівний відділ виконавчого комітету Кременчуцької міської ради/Р",
    "Архів:ДАЖО/Д",
    "Архів:ДАДнО/Р-6478/2", 
    "Архів:ДАЖО/752", 
    "Архів:ДАКрО/225/1/25", 
    "Архів:ДАКрО/225", 
    "Архів:ДАСО/Р", 
    "Архів:ДАХмО/К", 
    "Архів:ДАКрО/225/1/144а", 
    "Архів:ДАПО/Р",
    "Архів:ДАХмО/Р-6193",
    "Архів:ДАКрО/П-5907/2Р",
    "Архів:ДАОО/Р-8085/1",
    "Архів:ДАКрО/225/1",
    "Архів:ДАПО/1072/1/1",
    "Архів:ДАПО/978/1/135",
    "Архів:ДАПО/978",
    "Архів:ДАДнО/К/Ф",
    "Архів:ДАЖО/Р-1",
    "Архів:ДАХмО/Р-6193/12/1–5340",
    ]

In [ ]:
from urllib.parse import urlparse
from pathlib import PurePosixPath

DOCUMENT_SUFFIXES = {
    "pdf", "djvu", "djv",
    "tif", "tiff", "jp2",
    "zip", "cbz", "cbr",
}

IMAGE_SUFFIXES = {
    "jpg", "jpeg", "png", "gif", "bmp", "webp"
}

def sniff_suffix(url_or_title: str) -> str | None:
    """
    Returns one of:
      - "document"
      - "image"
      - None (unknown / webpage)
    """
    # Remove query / fragment
    parsed = urlparse(url_or_title)
    path = parsed.path or url_or_title

    suffix = PurePosixPath(path).suffix.lower().lstrip(".")
    if not suffix:
        return None

    if suffix in DOCUMENT_SUFFIXES:
        return "document"

    if suffix in IMAGE_SUFFIXES:
        return "image"

    return None

In [ ]:
def extract_links(title):
    title = canonicalize_title(title)
    raw = get_links(title).get("parse", {})    
    internal_links = []
    category_links = []
    children = []
    parent = None
    for link in raw.get("links", []):
        other_title = link.get("*")
        if other_title:
            canonical_title = canonicalize_title(other_title)
            item = {
                "title": canonical_title,
                "exists": "exists" in link
            }
            if canonical_title.startswith(title):
                children.append(item)
            elif title.startswith(canonical_title):
                parent = item
            elif _is_category_link(other_title):
                category_links.append(item)
            else:
                item["doc_type"] = sniff_suffix(canonical_title)
                internal_links.append(item)

    interwiki_links = []
    commons_links = []
    for link in raw.get("iwlinks", []):
        other_title = link.get("*")
        url = link.get("url")
        if other_title:
            item = {
                "title": other_title,
                "url": unquote(url),
                "doc_type": sniff_suffix(url),
            }
            if url.startswith("https://commons.wikimedia.org"):
                commons_links.append(item)
            else:
                interwiki_links.append(item)

    return {
        "title": title,
        "pageid": raw.get("pageid"),
        "parent": parent,
        "children": children,
        "category_links": category_links,
        "internal_links": internal_links,
        "commons_links": commons_links,
        "interwiki_links": interwiki_links,
        "external_links": raw.get("externallinks", []),
        #"raw": raw,
    }
            

In [ ]:
links = {
    title: extract_links(title)
    for title in titles
}

In [ ]:
extract_links("Архів:ДАКрО/225")

In [ ]:
#links["Архів:ДАХмО/Р-6193/12/1–5340"]

In [ ]:
def save_page(title):
    fname = title.replace("/", "_")
    fname = f"./var/{fname}.json"
    with open(fname, "w") as file:
        file.write(json.dumps(mw_read_page(title)))

def load_page(title):
    fname = title.replace("/", "_")
    fname = f"./var/{fname}.json"
    with open(fname) as file:
        return json.loads(file.read())

In [ ]:
def itemize_links(link_dict, link_expander=None):
    def _form_item(x, key):
        expander = link_expander.get(key, lambda x: x)
        result = {"text": form_text_item(x)}
        result["link"] = expander(x)
        return result
    result = {}
    for key, value in link_dict.items():
        result[key] = [_form_item(item, key) for item in value]
    return result

In [ ]:
def itemize_page(pg):
    def _expand_target(target):
        return _expand_link_target(target, title)
    _expander = {
        "category_links": _expand_target,
        "internal_links": _expand_target,
        }
    for key in ["notes", "other_links"]:
        pg[key] = itemize_links(pg[key], link_expander=_expander)
        #print(f"{key}:\n {pg[key]}")
    return pg

In [ ]:
import difflib

def edit_distance(a, b):
    matcher = difflib.SequenceMatcher(None, a, b)
    return int(round((1 - matcher.ratio()) * max(len(a), len(b))))

def link_array_item_difference(a, b):
    return edit_distance(a["link"], b["link"])

def compare_dict_arrays(a, b, diff_fn, threshold):
    result = [None] * len(a)
    matched_b = [False] * len(b)

    # Pass 1: Exact matches
    for i in range(len(a)):
        if i < len(b) and diff_fn(a[i], b[i]) == 0:
            result[i] = "equal"
            matched_b[i] = True

    # Pass 2: Edits and adds for unmatched segments
    i, j = 0, 0
    while i < len(a):
        if result[i] == "equal":
            i += 1
            j += 1
            continue
        # Advance j to next unmatched b[j]
        while j < len(b) and matched_b[j]:
            j += 1

        if j >= len(b):
            result[i] = "added"
            i += 1
            continue
        delta = diff_fn(a[i], b[j])
        if delta <= threshold:
            result[i] = "edited"
            matched_b[j] = True
            i += 1
            j += 1
        else:
            result[i] = "added"
            i += 1
    return result

In [ ]:
def compare_page_link_arrays(page, reference):
    def _map_comparison(comp):
        return None if comp == 'equal' else comp
        
    for key in "notes", "other_links":
        if key in page:
            if key in reference:
                #print("--- comparing:", key)
                for target_array, ref_array in zip(page[key].values(), reference[key].values()):
                    comparison = compare_dict_arrays(target_array, ref_array, link_array_item_difference, 1)
                    comparison = [_map_comparison(comp) for comp in comparison]
                    #print("====>", target_array, comparison)
                    for value, comp in zip(target_array, comparison):
                        value["edit"] = comp
                        value["link_edit"] = comp
            else:
                for link_array in page[key].values():
                    #print(link_array)
                    for i, value in enumerate(link_array):
                        #print(value)
                        value["edit"] = "added"
                        value["link_edit"] = "added"

In [ ]:
for title in titles:
    print("----", title, "----")
    pg = itemize_page(mw_read_page(title))
    for key in ["notes", "other_links"]:
        print(f"--------- {key} ----:")
        print(pg[key])

In [ ]:
pg = load_page(titles[3])
pg1 = itemize_page(deepcopy(pg))
del pg["notes"]
del pg["other_links"]
#pgi

In [ ]:
#compare_page_link_arrays(pg1, pg)
#for key in ["notes", "other_links"]:
#    print(key, pgi[key])
#pgi

In [ ]:
pg2 = deepcopy(pg1)
pg2["notes"]["commons_links"][0]["link"] = "https://commons.wikimedia.org/wiki/File:ДАКрО_225-1-25_Про_здійснення_закладної_кріпості_Х.Г.Г.Ґрітберґа_та_Л.Н.Луценка._(1919).pdf"

In [ ]:
for key in ["notes", "other_links"]:
    print(key, pg2[key])
print("##########")
compare_page_link_arrays(pg2, pg1)
print("##########")
for key in ["notes", "other_links"]:
    print(key, pg2[key])

In [ ]:
pg1["other_links"]

In [ ]:
pg1["notes"]

In [ ]:
#pg = load_page(titles[0])
ref_pg = deepcopy(pg)
pg["description"] = form_text_item("changed description")
pg["children"][0][0]["link"] = "changed link"
pg["children"][0][1]["link"] = "new link"
pg["children"][1][1]["text"] = form_text_item("changed child description")
#print(pg["children"][1])
check_page_changes(pg, ref_pg, report=True)
#print(pg["children"][1])

In [ ]:
compare_dict_arrays(pg["other_links"]["category_links"], pg["other_links"]["category_links"], link_array_item_difference, 1)

In [ ]:
compare_dict_arrays(pg["other_links"]["category_links"], pg["notes"]["category_links"], link_array_item_difference, 1)

In [ ]:
archive = Archive('DAZHO', 'D')
print(archive.title, get_title(archive.title), get_title(archive.url))
fond = archive['1']
print(fond.title, get_title(fond.title), get_title(fond.url))
opus = fond['1']
print(opus.title, get_title(opus.title), get_title(opus.url))
case = opus['376']
print(case.title, get_title(case.title), get_title(case.url))


In [ ]:
get_title(archive.title)

In [ ]:
get_title(archive.url)

In [ ]:
archive.history(limit=5)

In [ ]:
get_title("%D0%90%D1%80%D1%85%D1%96%D0%B2%3A%D0%94%D0%90%D0%96%D0%9E/%D0%94")

In [ ]:
#read_page(opus.url)

In [ ]:
#mw_read_page(get_title(opus.url))

In [ ]:
#mw_read_page("Архів:ДАПО/978/1", oldid="748029")["lastmod"]

In [ ]:
#mw_read_page("Архів:ДАПО/978/1")["lastmod"]

In [ ]:
#current = mw_read_page("Архів:ДАПО/978/1")
#reference = mw_read_page("Архів:ДАПО/978/1", oldid="748029")
#check_page_changes(current, reference)

In [ ]:
for title in titles:
    print("---", title)
    save_page(title)

In [ ]:
import re 

url = "https://uk.wikisource.org/w/index.php?title=%D0%90%D1%80%D1%85%D1%96%D0%B2%3A%D0%94%D0%90%D0%96%D0%9E/%D0%94&oldid=632022"

match = re.search(r"[?&]oldid=(\d+)", url)
if match:
    oldid = match.group(1)
    print(oldid)  # Output: 632022

In [ ]:
pg = mw_read_page(titles[1])
pg["children"][0]

In [ ]:
pg["header"]

In [ ]:
pg["notes"]

In [ ]:
pg["description"]

In [ ]:
pg=mw_read_page(titles[1])
pg["children"][:5]

In [ ]:
pg=mw_read_page(titles[2])

In [ ]:
pg["children"]

In [ ]:
wt = _read_wiki_text(titles[0])

In [ ]:
def process_title(title):
    wikitext, revid, title = _read_wiki_text(title)
    wikicode = mwparserfromhell.parse(wikitext)

    # Table data
    tables = [t for t in wikicode.filter_tags() if _is_table(t)]
    header = []
    children = []
    all_page_links = set()

    if tables:
        table_code = tables[0].contents # assume first table
        print(table_code)
        header, children = _parse_wikitext_table(table_code)

    print(f"------ {title} ({len(children)} rows)")
    print(f"{ARCHIVE_BASE}/wiki/{WIKI_NAMESPACE}:{title.replace(' ', '_')}")
    print("header:", header)
    if children:
        print("child[0]:", children[0])

In [ ]:
process_title(titles[0])

In [ ]:
import re

def tokenize_wikitext_table_line(text):
    token_re = re.compile(r'''
        (\[\[.*?\]\])       |  # group 1: wikilink
        (\|\||\!\!)         |  # group 2: table cell separators
        ([^|\[\]!]+)           # group 3: everything else
    ''', re.VERBOSE)

    return [m.group(0) for m in token_re.finditer(text)]


In [ ]:
test1="|[[Архів:ДАПО/978/1/1|1]]||Алфавітний список рекрутів, прийнятих за 96-м рекрутським набором||4 березня - 14 квітня 1831||31||[https://www.familysearch.org/records/images/search-results?imageGroupNumbers=108149209  108149209]"


In [ ]:
tokenize_wikitext_table_line(test1)

In [ ]:
pg = mw_read_page("https://uk.wikisource.org/wiki/Архів:ДААРК")

In [ ]:
link_target = "/5624 /"
target_parts = link_target.strip("/").split("/")
target_parts = [part.strip() for part in target_parts]
target_parts

In [ ]:
_expand_link_target("/5624 /", "Архів:ДАКО/782/1")

In [ ]:
pg=mw_read_page("Архів:ДАХмО/Р-6193/12/П23416")
pg

In [ ]:
txt=_read_wiki_text("Архів:ДАХмО/Р-6193/12/П23416")

In [ ]:
batch_fetch_document_links(["Архів:ДАХмО/Р-6193/12/П23416"])

In [ ]:
from birddog.wiki import _collect_doc_links_from_page, _file_link_to_url, _deduplicate_links

In [ ]:
_collect_doc_links_from_page({'title': {'uk': 'Архів:ДАХмО/Р-6193/12/П23416'}, 'template': {'uk': 'Архіви/справа'}, 'revid': None, 'description': {'uk': 'Мартинчук-Васик Онисія Харитонівна, 1885 р.н., с. Рудня-Новенька Шепетівського району'}, 'dates': {'uk': '1937–1989', 'en': '1937–1989'}, 'notes': {'commons_links': ['https://commons.wikimedia.org/wiki/File:ДАХмО_Р-6193-12-П23416._1937-1989._Мартинчук-Васик_Онисія_Харитонівна,_1885_р.н.,_с._Рудня-Новенька_Шепетівського_району.pdf']}, 'other_links': {'commons_links': [], 'category_links': ['Категорія:Хмельницька область'], 'internal_links': [], 'external_links': []}, 'tables': [], 'link': 'https://uk.wikisource.org/wiki/Архів:ДАХмО/Р-6193/12/П23416', 'doc_link': 'https://commons.wikimedia.org/wiki/File:ДАХмО_Р-6193-12-П23416._1937-1989._Мартинчук-Васик_Онисія_Харитонівна,_1885_р.н.,_с._Рудня-Новенька_Шепетівського_району.pdf'})

In [ ]:
links=['https://commons.wikimedia.org/wiki/File:ДАХмО_Р-6193-12-П23416._1937-1989._Мартинчук-Васик_Онисія_Харитонівна,_1885_р.н.,_с._Рудня-Новенька_Шепетівського_району.pdf']

In [ ]:
_file_link_to_url(links[0])

In [ ]:
txt=_read_wiki_text("Архів:ДАХмО/Р-6193/12/1–5340")

In [ ]:
pg = mw_read_page("Архів:ДАХмО/Р-6193/12/1–5340")